# SIB - Extra: distâncias Manhattan e Minkowski no KNN

O `KNNClassifier` e o `KNNRegressor` recebem a função de distância como parâmetro (`distance`),
por isso basta implementar novas funções com a mesma assinatura `(x, y) -> np.ndarray`.

| distância | fórmula |
|---|---|
| Euclidiana (L2) | $\sqrt{\sum_i (x_i - y_i)^2}$ |
| Manhattan (L1) | $\sum_i |x_i - y_i|$ |
| Minkowski (Lp) | $\left( \sum_i |x_i - y_i|^p \right)^{1/p}$ |

A Minkowski generaliza as outras duas: p=1 dá Manhattan, p=2 dá Euclidiana.

In [1]:
from functools import partial

import numpy as np
import matplotlib.pyplot as plt

from si.io.csv_file import read_csv
from si.model_selection.split import stratified_train_test_split
from si.models.knn_classifier import KNNClassifier
from si.statistics.euclidean_distance import euclidean_distance
from si.statistics.manhattan_distance import manhattan_distance
from si.statistics.minkowski_distance import minkowski_distance

## 1. Verificação das fórmulas

In [2]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([[1.0, 2.0, 3.0],
              [4.0, 6.0, 8.0],
              [0.0, 0.0, 0.0]])

print('euclidean       ', np.round(euclidean_distance(x, y), 4))
print('manhattan       ', np.round(manhattan_distance(x, y), 4))
print('minkowski p=1   ', np.round(minkowski_distance(x, y, p=1), 4))
print('minkowski p=2   ', np.round(minkowski_distance(x, y, p=2), 4))
print('minkowski p=3   ', np.round(minkowski_distance(x, y, p=3), 4))
print()
print('p=1 == manhattan:', np.allclose(minkowski_distance(x, y, p=1), manhattan_distance(x, y)))
print('p=2 == euclidean:', np.allclose(minkowski_distance(x, y, p=2), euclidean_distance(x, y)))

euclidean        [0.     7.0711 3.7417]
manhattan        [ 0. 12.  6.]
minkowski p=1    [ 0. 12.  6.]
minkowski p=2    [0.     7.0711 3.7417]
minkowski p=3    [0.     6.     3.3019]

p=1 == manhattan: True
p=2 == euclidean: True


## 2. Usar a Minkowski no KNN

O KNN chama `self.distance(sample, self.dataset.X)` com dois argumentos, por isso o `p` tem
de ser fixado antes. `functools.partial` resolve isso.

In [3]:
iris = read_csv('../datasets/iris/iris.csv', sep=',', features=True, label=True)
train, test = stratified_train_test_split(iris, test_size=0.3, random_state=42)

print('treino:', train.shape(), '| teste:', test.shape())

treino: (105, 4) | teste: (45, 4)


In [4]:
distances = {
    'euclidean': euclidean_distance,
    'manhattan': manhattan_distance,
    'minkowski p=3': partial(minkowski_distance, p=3),
    'minkowski p=10': partial(minkowski_distance, p=10),
}

print('{:<16s} {:>10s} {:>10s}'.format('distância', 'k=1', 'k=5'))
for name, distance in distances.items():
    scores = [KNNClassifier(k=k, distance=distance).fit(train).score(test) for k in (1, 5)]
    print('{:<16s} {:>10.4f} {:>10.4f}'.format(name, *scores))

distância               k=1        k=5
euclidean            0.9556     0.9556
manhattan            0.9556     0.9556
minkowski p=3        0.9556     0.9556
minkowski p=10       0.9556     0.9556


## 3. Efeito do p ao longo de vários k

In [5]:
ks = range(1, 16)
plt.figure(figsize=(7, 4))
for name, distance in distances.items():
    accuracies = [KNNClassifier(k=k, distance=distance).fit(train).score(test) for k in ks]
    plt.plot(list(ks), accuracies, marker='o', label=name)
plt.xlabel('k')
plt.ylabel('accuracy (teste)')
plt.title('KNN no iris com diferentes distâncias')
plt.legend()
plt.tight_layout()
plt.show()

## Conclusões

- No iris as três distâncias dão resultados praticamente iguais: com 4 features numéricas
  e na mesma escala, a escolha da métrica pesa pouco.
- A diferença aparece em dimensões altas, onde a L1 (Manhattan) costuma discriminar melhor
  do que a L2 e do que valores de p elevados.
- Com p a crescer, a Minkowski aproxima-se da distância de Chebyshev (máximo das diferenças).